## Working with BM-25(Best Matching 25) algorithm

It's simply an improvement on TF-IDF algorithm, made efficient for word normalization.

### Term Frequency (TF)

The number of times a particular word appears in a document. (how often a query appears in a document)

### Inverse Document Frequency (IDF)

The number of times a word appears in a list of documents. (measures the importance of a term across the entire corpus)

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

PDF_LIST = ["../docs/crime_act.pdf", "../docs/interim_government_act.pdf", "../docs/labor_act.pdf"]

RAW_KNOWLEDGE_BASE = []

for pdf_path in PDF_LIST:
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()

    for page in pages:
        RAW_KNOWLEDGE_BASE.append(
            Document(
                page_content=page.page_content,
                metadata={
                    "source": pdf_path,
                    "page": page.metadata["page"]+1
                }
            )
        )

print(f"Total pages loaded: {len(RAW_KNOWLEDGE_BASE)}")

Total pages loaded: 141


### For chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer
from typing import List, Optional
from langchain_core.documents import Document

EMBEDDING_MODEL_NAME = "thenlper/gte-small"

MARKDOWN_SEPARATORS = [
    "\n#{1,6} ",
    "```\n",
    "\n\\*\\*\\*+\n",
    "\n---+\n",
    "\n___+\n",
    "\n\n",
    "\n",
    " ",
    "",
]

def split_documents(
    chunk_size: int,
    knowledge_base: List[Document],
    tokenizer_name: Optional[str] = EMBEDDING_MODEL_NAME,
) -> List[Document]:
    """
    Split documents into chunks of maximum size `chunk_size` tokens and return a list of documents.
    """
    text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
        AutoTokenizer.from_pretrained(tokenizer_name),
        chunk_size=chunk_size,
        chunk_overlap=int(chunk_size / 10),
        add_start_index=True,
        strip_whitespace=True,
        separators=MARKDOWN_SEPARATORS,
    )

    docs_processed = []
    for doc in knowledge_base:
        docs_processed += text_splitter.split_documents([doc])

    # Remove duplicates
    unique_texts = {}
    docs_processed_unique = []
    for doc in docs_processed:
        if doc.page_content not in unique_texts:
            unique_texts[doc.page_content] = True
            docs_processed_unique.append(doc)

    return docs_processed_unique

In [4]:
docs_processed = split_documents(
    512,  # We choose a chunk size adapted to our model
    RAW_KNOWLEDGE_BASE,
    tokenizer_name=EMBEDDING_MODEL_NAME,
)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)

def tokenize_for_bm25(text: str):
    return tokenizer.tokenize(text.lower())

In [ ]:
tokenized_docs = [
    tokenize_for_bm25(doc.page_content)
    for doc in docs_processed
]

In [1]:
import math
from collections import defaultdict, Counter

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.docs = docs
        self.N = len(docs)
        self.avgdl = sum(len(doc) for doc in self.docs) / self.N
        self.k1 = k1
        self.b = b

        self.df = defaultdict(int)
        for doc in self.docs:
            for word in set(doc):
                self.df[word] += 1

        self.idf = {
            word: math.log((self.N - freq + 0.5)/(freq + 0.5))
            for word, freq in self.df.items()
        }
    
    def score(self, query, doc_idx):
        doc = self.docs[doc_idx]
        freq = Counter(doc)
        score = 0

        for term in query.split():
            if term not in freq:
                continue
            f = freq[term]
            idf = self.idf.get(term, 0)
            denom = f + self.k1 * (1 - self.b + self.b * len(doc) / self.avgdl)
            score += idf * (f * (self.k1 + 1)) / denom
        return score
    
    def search(self, query, top_k=5):
        scores = [(i, self.score(query, i)) for i in range(self.N)]
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

In [ ]:
bm25 = BM25(tokenized_docs)

query = "What is the meaning of basic remuneration?"

top_docs = bm25.search(query, top_k=7)

In [ ]:
results = [docs_processed[i] for i, _ in top_docs]
results

[Document(metadata={'source': '../docs/crime_act.pdf', 'page': 1}, page_content='1 \nThe Criminal Offences (Sentencing and \nExecution) Act, 2017 (2074)  \n \nDate of Authentication:  \n16 October 2017 (2074.6.30) \n \nAct Number 38 of the year 2017 (2074) \nAn Act Made to Provide for Determination and Execution of \nSentences for Criminal Offences \n \nPreamble: Whereas it is expedient to make legal provisions for the \ndetermination of appropriate sentences for offenders who commit \ncriminal offences and the execution of such sentences, in order to \nmaintain the interest and decency of the general public by creating a just, \npeaceful and safe society;  \nNow, therefore, the Legislature -Parliament under clause (1) of \nArticle 296 of the Constitution of Nepal has made this Act. \nChapter – 1 \nPreliminary \n1. Short title and commencement: (1) This Act is cited as the \n"Criminal Offenses ( Sentencing and Execution) Act, 2017 \n(2074)". \n    (2) Clauses (d), (e), (f) and (g) of S

## Vector database

In [1]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores.utils import DistanceStrategy

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    multi_process=False,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

KNOWLEDGE_VECTOR_DATABASE = FAISS.from_documents(
    docs_processed, embedding_model, distance_strategy=DistanceStrategy.COSINE
)

NameError: name 'EMBEDDING_MODEL_NAME' is not defined

In [ ]:
question = "What is the meaning of 'Basic remuneration'?"
query_vector = embedding_model.embed_query(question)

In [ ]:
print(f"\nStarting retrieval for {question=}...")
retrieved_docs = KNOWLEDGE_VECTOR_DATABASE.similarity_search(query=question, k=5)

### Reranking

In [ ]:
from rerankers import Reranker
reranker = Reranker(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

### Add the documents obtained from BM-25 and Embedding vector search

In [ ]:
final_docs = [doc.page_content for doc in results]
retrieval = [doc.page_content for doc in results]
final_docs.extend(retrieval)

In [ ]:
num_docs_final = 5

if reranker:
    print("===> Reranking documents...")
    doc_texts = [doc.page_content for doc in final_docs]
    rerank_results = reranker.rank(question, doc_texts)
    reranked_docs = []

    for res in rerank_results.results[:num_docs_final]:
        for doc in final_docs:
            if doc.page_content == res.document:
                reranked_docs.append(doc)
                break
else:
    relevant_docs = final_docs[:num_docs_final]